# 🤖 Project 9.2 — Robot Task Scheduler (Topological Sort)
**DSA for Mechatronics · Week 09 — Graphs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict, deque, Counter
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A robot assembly cell executes **tasks** in a fixed sequence.
Some tasks depend on others being completed first — these are **prerequisites**.
This forms a **Directed Acyclic Graph (DAG)**.

We need to:
1. Find a valid execution order (**topological sort** — Kahn's algorithm)
2. Detect if a **deadlock** exists (cycle = impossible schedule)
3. Compute the **critical path** (longest dependency chain = minimum completion time)
4. Find **all valid orderings** using DFS backtracking


## Step 1 — Generate task dependency graph

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_TASKS    = random.randint(8, 14)
N_PREREQS  = random.randint(N_TASKS, N_TASKS + random.randint(2, 6))

TASK_TYPES = ["DRILL","WELD","INSPECT","BOLT","POLISH","PAINT",
              "ASSEMBLE","TEST","CALIBRATE","LOAD","UNLOAD",
              "SCAN","GRIP","CUT","PRESS"]
TASK_NAMES = [f"T{i+1:02d}_{TASK_TYPES[i % len(TASK_TYPES)]}" for i in range(N_TASKS)]
DURATIONS  = {i: random.randint(2, 12) for i in range(N_TASKS)}   # minutes

# Build a DAG: ensure no cycle by only adding edges u→v where u < v (topological order hint)
topo_order = list(range(N_TASKS))
random.shuffle(topo_order)
rank = {node: i for i, node in enumerate(topo_order)}   # rank[node] = position in true order

prereqs = set()
# ensure each node (except first) has at least one predecessor
for i in range(1, N_TASKS):
    u = topo_order[random.randint(0, i-1)]
    v = topo_order[i]
    prereqs.add((u, v))

# add extra edges (only rank[u] < rank[v] to keep DAG)
for _ in range(N_PREREQS - len(prereqs)):
    u = random.randint(0, N_TASKS-1)
    v = random.randint(0, N_TASKS-1)
    if rank[u] < rank[v]:
        prereqs.add((u, v))

prereqs = list(prereqs)

# Build adjacency list and in-degree map
dag  = defaultdict(list)
indeg = defaultdict(int)
for u, v in prereqs:
    dag[u].append(v)
    indeg[v]   # ensure exists
for i in range(N_TASKS):
    indeg[i]   # initialise

print(f"Task dependency graph:")
print(f"  Tasks          : {N_TASKS}")
print(f"  Dependencies   : {len(prereqs)}")
print()
print(f"  {'Task':<24}  {'Duration':>10}  {'Depends on'}")
print(f"  {'─'*24}  {'─'*10}  {'─'*30}")
prereq_map = defaultdict(list)
for u, v in prereqs:
    prereq_map[v].append(TASK_NAMES[u])
for i in range(N_TASKS):
    deps = prereq_map[i] if prereq_map[i] else ["—"]
    print(f"  {TASK_NAMES[i]:<24}  {DURATIONS[i]:10d}  {deps}")


## Step 2 — Kahn's algorithm (topological sort + deadlock detection)

In [ ]:
def kahns_topological_sort(dag, indeg, n):
    """
    Kahn's algorithm — topological sort via BFS on in-degrees.

    Algorithm:
      1. Enqueue all nodes with in-degree 0 (no prerequisites).
      2. While queue is not empty:
         a. Dequeue node u — add to result order.
         b. For each successor v: decrement indeg[v].
            If indeg[v] == 0: enqueue v (all its prerequisites done).
      3. If result length < n: a cycle exists (deadlock detected).

    O(V + E) time.
    """
    # work on copies
    indeg_copy = dict(indeg)
    queue      = deque([i for i in range(n) if indeg_copy[i] == 0])
    order      = []

    print(f"Kahn's algorithm step trace:")
    print(f"  {'Step':>5}  {'Dequeued':<20}  {'Queue after'}")
    print(f"  {'─'*5}  {'─'*20}  {'─'*30}")

    step = 0
    while queue:
        u = queue.popleft()
        order.append(u)
        step += 1
        for v in sorted(dag[u]):
            indeg_copy[v] -= 1
            if indeg_copy[v] == 0:
                queue.append(v)
        q_str = str([TASK_NAMES[x] for x in list(queue)[:4]])
        print(f"  {step:5d}  {TASK_NAMES[u]:<20}  {q_str}")

    has_cycle = len(order) < n
    return order, has_cycle

exec_order, deadlock = kahns_topological_sort(dag, indeg, N_TASKS)

print()
if deadlock:
    print("⚠  DEADLOCK DETECTED — cycle in dependency graph!")
    print("   Some tasks can never be scheduled.")
else:
    print(f"✅ Valid execution order found ({N_TASKS} tasks):")
    for rank_i, task_i in enumerate(exec_order, 1):
        print(f"   {rank_i:3d}. {TASK_NAMES[task_i]}  ({DURATIONS[task_i]} min)")


## Step 3 — Critical path (longest dependency chain)

In [ ]:
def critical_path(dag, exec_order, durations, n):
    """
    Compute the earliest start time for each task using the topological order.
    earliest[v] = max(earliest[u] + duration[u]) for all u → v.
    The critical path length = max(earliest[v] + duration[v]).
    Also track which path achieves the maximum.
    """
    earliest = [0] * n
    for u in exec_order:
        for v in dag[u]:
            if earliest[u] + durations[u] > earliest[v]:
                earliest[v] = earliest[u] + durations[u]

    # Find critical path by backtracking from the last task
    finish_times = [earliest[i] + durations[i] for i in range(n)]
    total_time   = max(finish_times)
    critical_end = finish_times.index(total_time)

    # Reconstruct critical path (greedy: at each node pick predecessor with max finish)
    rev_dag = defaultdict(list)
    for u, v in prereqs:
        rev_dag[v].append(u)

    path = [critical_end]
    while rev_dag[path[-1]]:
        predecessors = rev_dag[path[-1]]
        best_pred    = max(predecessors, key=lambda u: earliest[u] + durations[u])
        # only follow if on critical path
        if earliest[best_pred] + durations[best_pred] == earliest[path[-1]]:
            path.append(best_pred)
        else:
            break
    path.reverse()

    return earliest, finish_times, total_time, path

earliest, finish_times, total_time, crit_path = critical_path(dag, exec_order, DURATIONS, N_TASKS)

print(f"Task schedule (earliest start times):")
print(f"  {'Task':<24}  {'Earliest start':>14}  {'Finish':>8}  {'On critical path':>18}")
print(f"  {'─'*24}  {'─'*14}  {'─'*8}  {'─'*18}")
for i in exec_order:
    on_crit = "★ YES" if i in crit_path else ""
    print(f"  {TASK_NAMES[i]:<24}  {earliest[i]:14d}  {finish_times[i]:8d}  {on_crit:>18}")

print(f"\n  Total minimum completion time : {total_time} min")
crit_str = " → ".join(TASK_NAMES[i] for i in crit_path)
print(f"  Critical path                 : {crit_str}")
print(f"  Critical path length          : {len(crit_path)} tasks")


## Step 4 — Cycle detection in a modified (cyclic) version

In [ ]:
def detect_cycle_directed(adj_dir, n):
    """
    Detect cycle in directed graph using DFS with 3-colour marking.
    WHITE = 0 (unvisited), GRAY = 1 (in current DFS stack), BLACK = 2 (done).
    A back edge (GRAY node encountered) = cycle.
    Returns (has_cycle, cycle_nodes).
    """
    color     = [0] * n
    parent    = [-1] * n
    cycle_end = [None]

    def dfs(u):
        color[u] = 1   # GRAY — currently exploring
        for v in adj_dir[u]:
            if color[v] == 0:
                parent[v] = u
                if dfs(v): return True
            elif color[v] == 1:   # back edge = cycle
                cycle_end[0] = (u, v)
                return True
        color[u] = 2   # BLACK — fully processed
        return False

    for start in range(n):
        if color[start] == 0:
            if dfs(start): return True, cycle_end[0]
    return False, None

# Original DAG — should be cycle-free
has_cycle_dag, _ = detect_cycle_directed(dag, N_TASKS)

# Inject one back-edge to create a cycle for demonstration
u_cyc = exec_order[-1]
v_cyc = exec_order[1]   # last task points back to second task
cyclic_dag = defaultdict(list, {k: list(v) for k, v in dag.items()})
cyclic_dag[u_cyc].append(v_cyc)

has_cycle_cyc, cycle_edge = detect_cycle_directed(cyclic_dag, N_TASKS)

print(f"Cycle detection:")
print(f"  Original DAG has cycle    : {'YES ⚠' if has_cycle_dag else 'NO ✅ (valid DAG)'}")
print()
print(f"  After injecting back-edge {TASK_NAMES[u_cyc]} → {TASK_NAMES[v_cyc]}:")
print(f"  Modified graph has cycle  : {'YES ⚠ (deadlock!)' if has_cycle_cyc else 'NO'}")
if cycle_edge:
    eu, ev = cycle_edge
    print(f"  Back edge detected        : {TASK_NAMES[eu]} → {TASK_NAMES[ev]}")
    print(f"  (This creates a circular dependency — robot would loop forever.)")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROBOT TASK SCHEDULER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  TASK GRAPH" + " "*(W-12) + "║")
print(f"║  {'Tasks (vertices)':<28}: {N_TASKS:<26}║")
print(f"║  {'Dependencies (edges)':<28}: {len(prereqs):<26}║")
print(f"║  {'Deadlock (cycle)':<28}: {'YES ⚠' if deadlock else 'NO ✅':<26}║")
print("╠" + "═"*W + "╣")
print("║  TOPOLOGICAL ORDER (first 5)" + " "*(W-30) + "║")
for rank_i, task_i in enumerate(exec_order[:5], 1):
    print(f"║  {'  '+str(rank_i)+'. '+TASK_NAMES[task_i]:<28}  {DURATIONS[task_i]} min{'':<21}║")
print("╠" + "═"*W + "╣")
print("║  CRITICAL PATH" + " "*(W-15) + "║")
print(f"║  {'Minimum completion time':<28}: {total_time} min{'':<20}║")
print(f"║  {'Critical path length':<28}: {len(crit_path)} tasks{'':<20}║")
crit_str_short = "→".join(TASK_NAMES[i].split("_")[0] for i in crit_path)
print(f"║  {'Critical path':<28}: {crit_str_short[:26]:<26}║")
print("╠" + "═"*W + "╣")
print("║  CYCLE DETECTION" + " "*(W-17) + "║")
print(f"║  {'Original DAG':<28}: {'NO CYCLE ✅' if not has_cycle_dag else 'CYCLE ⚠':<26}║")
print(f"║  {'After back-edge injection':<28}: {'CYCLE DETECTED ⚠' if has_cycle_cyc else 'NO CYCLE':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the network or system?

*Your answer here:*

---

**Q2 — Which graph concept did you find most important, and why?**
Pick one algorithm (BFS, DFS, topological sort, Dijkstra, cycle detection…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
